# Bronze to Silver: Data Cleaning and Transformation for Dimension Tables

This notebook performs the silver-layer processing for all dimension tables in the e-commerce medallion architecture. It reads raw bronze tables, applies data quality checks, cleanses and standardizes values, and writes the transformed dimension tables to the silver schema.

## Dimensions Processed

| Dimension | Bronze Table | Silver Table | Rows | Key Transformations |
| --- | --- | --- | --- | --- |
| Brands | `bronze.brz_brands` | `silver.slv_brands` | 52 | Remove `@` from brand codes, trim whitespace, standardize category codes |
| Category | `bronze.brz_category` | `silver.slv_category` | 8 | Uppercase category codes, remove duplicate rows |
| Customers | `bronze.brz_customers` | `silver.slv_customers` | 299,700 | Drop null customer IDs, fill null phones, format phone numbers |
| Date | `bronze.brz_date` | `silver.slv_date` | 92 | Cast date/year to proper types, title-case day names, create readable quarter/week labels, remove duplicates |
| Products | `bronze.brz_products` | `silver.slv_products` | 50,000 | Fill null colors, cast numeric columns, uppercase codes, fix material spelling, fix negative ratings |

All tables are in the `ecommerce` catalog.

## Standard Processing Pattern

Each dimension follows the same pipeline:

1. **Read** from the bronze Delta table
2. **Check nulls** — count null values per column to identify data quality issues
3. **Inspect distinct values** — display distinct values to spot inconsistencies, formatting issues, or duplicates
4. **Clean and transform** — apply column-specific fixes (trimming, casing, type casting, spelling corrections, etc.)
5. **Check duplicates** — verify no duplicate keys exist; remove if found
6. **Write to Silver** — save the cleaned DataFrame as a managed Delta table with `mode("overwrite")`

**Basic Data Cleaning Process**

```text
Raw Data
   ↓
Schema validation
   ↓
Normalize whitespaces
   ↓
NULL validation
   ↓
Duplicate validation
   ↓
Data type / format validation
   ↓
Business-rule validation
   ↓
Clean Data
```

In [0]:
# Import required libraries for data cleaning and transformation

import pyspark.sql.functions as F
from pyspark.sql.functions import to_date, initcap
from pyspark.sql.types import IntegerType, FloatType

In [0]:
# Define the catalog name for the e-commerce project 
catalog_name = 'ecommerce'

### Brands Dim Processing

Cleans the brands dimension by removing non-alphanumeric characters from `brand_code`, trimming whitespace from `brand_name`, and standardizing `category_code` values to short-form codes. No nulls or duplicates were found.

In [0]:
# Read the brands dimension table from the bronze layer
df_brands = spark.read.table(f"{catalog_name}.bronze.brz_brands")

df_brands.show(5)

In [0]:
# Check for null values in each column of the brands table

df_brands.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_brands.columns
]).show()

In [0]:
# Display distinct brand codes to identify any data quality issues
df_brands.select('brand_code').distinct().show()

**Cleaning Brand Codes:**

In the distinct list of brand codes, we can see that some values contain `@` characters. We need to remove those, as brand codes should only contain alphanumeric characters. We will also trim any extra leading or trailing whitespace in the brand codes.

**Steps:**
1. Use `regexp_replace` to remove all `@` characters from `brand_code`.
2. Use `trim` to remove any leading or trailing whitespace.
3. Verify the cleaned values with a distinct listing.

In [0]:
# Remove non-alphanumeric characters from brand_code and trim whitespace
df_brands = df_brands.withColumn('brand_code', F.trim(F.regexp_replace(F.col('brand_code'), r'[^A-Za-z0-9]', '')))

df_brands.select('brand_code').distinct().show()


In [0]:
# Display distinct brand names to check for whitespace issues
df_brands.select('brand_name').distinct().show()

<b>Cleaning Brand Names:</b>

In the distinct listing of `brand_name` values, we can see some entries contain leading or trailing whitespace. We will remove those using `trim`, the same approach we used for `brand_code` in the previous step.

Steps:
1. Use `trim` to remove any leading or trailing whitespace from `brand_name`.
2. Verify the cleaned values with a distinct listing.

In [0]:
# Trim leading and trailing whitespace from brand_name
df_brands = df_brands.withColumn('brand_name', F.trim('brand_name'))

df_brands.select('brand_name').distinct().show()

In [0]:
# Display distinct category codes to identify inconsistencies
df_brands.select('Category_code').distinct().show()

**Cleaning Category Codes:**

In the above output we can see the distinct values in the `category_code` column. There are a few repetitions of codes in both full form and short form — for example, `'BKS'` and `'BOOKS'` both represent books. We will standardize these so that only one category code is kept for each category.

To replace we will use `.replace()` and we will pass a dictionary of changes at one go.

**Steps:**
1. Use `.replace()` with a dictionary mapping full-form codes to their standardized short-form equivalents.
2. Verify the cleaned values with a distinct listing.

In [0]:
# Standardize category codes by mapping full-form names to short-form codes
replacements = {
    'GROCERY': 'GRCY',
    'BOOKS': 'BKS',
    'TOYS': 'TOY'
}

df_brands = df_brands.replace(to_replace=replacements, subset=['Category_code'])

df_brands.select('Category_code').distinct().show()
df_brands.select(F.countDistinct('Category_code')).show()

In [0]:
# checking for duplicates in brand_code
df_duplicate = df_brands.groupBy("brand_code").count().filter(F.col("count") > 1)
df_duplicate.show()


# checking for duplicates in brand_name
df_duplicate = df_brands.groupBy("brand_name").count().filter(F.col("count") > 1)
df_duplicate.show()

# Check for duplicate rows across all key columns
df_duplicate = (
    df_brands.groupBy(["brand_code", "brand_name", "Category_code"])
    .count()
    .filter(F.col("count") > 1)
)
df_duplicate.show()

**Save the cleaned Brands dimension table to the Silver layer**

All columns in the `brz_brands` table have now been cleaned and transformed:

- **`brand_code`** — Removed non-alphanumeric characters and trimmed whitespace.
- **`brand_name`** — Trimmed leading and trailing whitespace.
- **`Category_code`** — Standardized short-form and full-form codes to a single representation.

With the transformations complete, we can write the cleaned DataFrame to the Silver layer as a managed Delta table (`slv_brands`).

In [0]:
# Write the cleaned brands table to the silver layer
df_brands.write \
        .mode("overwrite") \
        .format("delta") \
        .option("mergeSchema", True) \
        .saveAsTable(f"{catalog_name}.silver.slv_brands")

### Category Dim Processing

Cleans the category dimension by converting `category_code` to uppercase for consistency with the brands table and removing duplicate rows based on `category_code`. No nulls were found.

In [0]:
# Read the category dimension table from the bronze layer
df_category = spark.read.table(f"{catalog_name}.bronze.brz_category")

df_category.show(5)

In [0]:
# Check for null values in each column of the category table
df_category.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_category.columns
]).show()

In [0]:
# Display distinct category codes and their count
df_category.select('category_code').distinct().show()

df_category.select(F.countDistinct('category_code')).show()

   
**Standardizing Category Code Casing:**

The category codes in the `df_category` table are correct, however they are in lowercase. To keep the same standards, we will convert these to uppercase so they match the `category_code` column in the brands table.

**Steps:**
1. Use `upper()` to convert all `category_code` values to uppercase.
2. Verify the transformed values with a distinct listing.

In [0]:
# Convert category_code values to uppercase for consistency
df_category = df_category.withColumn('category_code', F.upper('category_code'))

df_category.select('category_code').distinct().show()

In [0]:
# Display distinct category names to verify data quality
df_category.select('category_name').distinct().show(truncate=False)

In [0]:
# Check for duplicate rows in the category table
df_duplicate = df_category.groupBy('category_code', 'category_name').count().filter(F.col('count') > 1)
df_duplicate.show()

In [0]:
# Remove duplicate rows based on category_code
df_category = df_category.dropDuplicates(['category_code'])

**Save the cleaned Category dimension table to the Silver layer**

All columns in the `brz_category` table have now been cleaned and transformed:

- **`category_code`** — Converted to uppercase for consistency with the brands table.
- **`category_name`** — Verified for data quality; no issues found.
- **Duplicates** — Removed duplicate rows based on `category_code`.

With the transformations complete, we can write the cleaned DataFrame to the Silver layer as a managed Delta table (`slv_category`).

In [0]:
# Write the cleaned category table to the silver layer
df_category.write \
        .mode("overwrite") \
        .format("delta") \
        .option("mergeSchema", "true") \
        .saveAsTable(f"{catalog_name}.silver.slv_category")

### Customers Dim Processing

Cleans the customers dimension (300,000 rows) by dropping rows with null `customer_id` (300 rows, ~0.1%), filling null `phone` values with 'Not Available', and removing the `.0` suffix from phone numbers. Final row count: 299,700.

In [0]:
# Read the customers dimension table from the bronze layer
df_customers = spark.read.table(f'{catalog_name}.bronze.brz_customers')
df_customers.show(5)

In [0]:
# Get the row and column counts of the customers table
row_count, col_count = df_customers.count(), len(df_customers.columns)
print(f"Row count: {row_count}\nColumn count: {col_count}")

**Observations on the Customers table:**

- The `customer_id` column is alphanumeric.
- The `phone` column is currently stored as a float (e.g., ending in `.0`). We need to correctly format this — either by casting it to an integer type or by stripping the `.0` suffix and casting it to a string.

Before applying transformations, we will first check whether any null values are present in the table.

We will apply the following transformations one by one:

1. **Check for nulls** — Inspect each column for null or missing values so we can decide on an appropriate handling strategy.
2. **Format `phone`** — Remove the `.0` suffix and cast to the correct type.

In [0]:
# Check for null values in all columns of the customers table

df_customers.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_customers.columns
]).show()



**Handling Nulls in the Customers Table**

The null-count check reveals nulls in two columns:

- **`customer_id`** — This is a required field. To address the nulls, we would normally either drop the affected rows or consult the domain owner for guidance. In this case, we will drop the null rows since they account for only ~300 out of ~300,000 records — a negligible fraction that won't materially impact downstream analysis.
- **`phone`** — This is an optional field, so nulls here are acceptable and no action is needed.

**Steps:**
1. Drop rows where `customer_id` is null.
2. Verify the null counts after the cleanup.

In [0]:
# Drop rows where customer_id is null (required field, only ~300 of 300K rows)
df_customers = df_customers.dropna(subset=['customer_id'])

In [0]:
# Row count after dropping rows where customer_id is null
print(f"Current row count: {df_customers.count()}")

In [0]:
# Fill null phone values with 'Not Available' since phone is an optional field
df_customers = df_customers.fillna('Not Available', subset=['phone'])

All null values in the customers table have been handled. Let's proceed with the remaining transformations.

In [0]:
# Remove the '.0' suffix from phone numbers using regex
df_customers = df_customers.withColumn(
    'phone',
    F.regexp_replace(
        F.col("phone"),
        r"\..*$",
        ""
    )
)

df_customers.show(5)

In [0]:
# Checking for phone numbers that are not 12 digits long.
# A valid phone number is expected to be exactly 12 digits (e.g., with country code).
# We first display the invalid rows, then replace them with 'Not Available'.

# Display rows where the phone number length is not 12
# df_customers.filter(F.length(F.col("phone")) != 12).display()

# Replace invalid phone numbers (not 12 digits) with 'Not Available'
# df_customers = df_customers.withColumn(
#     "phone",
#     F.when(F.length(F.col("phone")) != 12, 'Not Available').otherwise(F.col("phone"))
# )


**Saving the Customers Silver Layer Table**

The data looks good — all nulls have been handled and the `phone` column has been properly formatted. We can now save the cleaned and transformed `df_customers` DataFrame as the customers silver layer table.

In [0]:
# Write the cleaned customers table to the silver layer
df_customers.write \
    .mode("overwrite") \
    .format("delta") \
    .option("mergeSchema", True) \
    .saveAsTable(f"{catalog_name}.silver.slv_customers")

### Date Dim Processing

Cleans the date dimension (95 rows) by casting `date` to a proper date type, `year` to integer, title-casing `day_name`, creating readable `quarter` (e.g., `Q3-2025`) and `week_of_year` (e.g., `Week31-2025`) labels with numeric companion columns, and removing 3 duplicate date rows. Final row count: 92.

In [0]:
# Read the date dimension table from the bronze layer
df_date = spark.read.table(f"{catalog_name}.bronze.brz_date")
df_date.show(5)

In [0]:
# Check for null values in each column of the date table

df_date.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_date.columns
]).show()

In [0]:
# Get the row and column counts of the date table
print(f"Current row count: {df_date.count()}")
print(f"Current column count: {len(df_date.columns)}")

In [0]:
# Display the schema of the date table

df_date.printSchema()

In [0]:
# Analyze the distribution of values using the describe method

df_date.describe().display()

In [0]:
# Check for duplicate dates in the date table

duplicates = df_date.groupBy("date").count().filter("count > 1")

print(f"Duplicate count: {duplicates.count()}")
duplicates.display()


**Observations on the Date table:**

From the schema and description we can identify the following transformations needed:

1. **`date` column** — Currently stored as a string; we need to cast it to a proper `date` type.
2. **`year` column** — Currently stored as a string; we need to cast it to an `int`.
3. **`day_name` column** — Day names are in inconsistent case; we need to convert everything to lowercase for consistency.
4. **`week_of_year` column** — From the description, all values are negative. We will convert them to positive values and rename the column to `week_number` in a single step — the positive numeric week value is preserved under a clearer name.
5. **`quarter` column** — We will rename the column to `quarter_number` in a single step — the original numeric quarter value is preserved under a clearer name.
6. **Enhanced `quarter` column** — Create a new `quarter` column with a readable format like `Q3-2025` (combining the quarter number with the year).
7. **Enhanced `week_of_year` column** — Create a new `week_of_year` column with a readable format like `Week12-2025` (combining the week number with the year).
8. **Remove duplicates** — The duplicate check found duplicate values in the `date` column; we need to drop duplicate rows to ensure each date is unique.

In [0]:
# Cast the 'date' column from string to a proper Spark date type.
# to_date takes the INPUT format of the string ("dd-MM-yyyy") so Spark knows how to parse it.
df_date = df_date.withColumn(
    "date",
    to_date(df_date["date"], "dd-MM-yyyy")
)

In [0]:
# Verify the date column cast and inspect the schema
df_date.printSchema()
df_date.show(5, truncate=False)

In [0]:
# 2. Cast the 'year' column from string to int.
# 3. Convert the 'day_name' column to title case (initcap) for consistency.
df_date = df_date.withColumn("year", df_date["year"].cast("int"))
df_date = df_date.withColumn("day_name", F.initcap(F.col("day_name")))

In [0]:
# 5. Rename the 'quarter' column to 'quarter_number' (preserving the numeric quarter value under a clearer name).
# 6. Create an enhanced 'quarter' column in a readable format like 'Q3-2025' (combining the quarter number with the year).

df_date = df_date.withColumn(
    "quarter_number",
    F.col("quarter").cast("int")
)

df_date = df_date.withColumn(
    "quarter",
    F.concat(F.lit("Q"), F.col("quarter"), F.lit("-"), F.col("year"))
)

In [0]:
# Display the date table after casting year and day_name
df_date.show(5)

In [0]:
# 4 & 7. Convert the 'week_of_year' column values from negative to positive,
#    rename it to 'week_number' (preserving the numeric week value under a clearer name),
#    and create a new enhanced 'week_of_year' column in a readable format like 'Week12-2025'
#    (combining the week number with the year).

df_date = df_date.withColumn(
    "week_number",
    F.abs(F.col("week_of_year")).cast("int")
)

df_date = df_date.withColumn(
    "week_of_year",
    F.concat(F.lit("Week"), F.col('week_number'), F.lit("-"), F.col("year"))
)

In [0]:
# Display the date table after transforming quarter and week columns
df_date.show(5)

In [0]:
# Display the final schema of the date table
df_date.printSchema()

In [0]:
# 8. Remove duplicate rows from the Date table to ensure each date is unique.

df_date = df_date.dropDuplicates(['date'])

print(f"Current row count: {df_date.count()}")

In [0]:
# Write the cleaned date table to the silver layer

df_date \
    .write.mode("overwrite") \
    .format("delta") \
    .option("mergeSchema", True) \
    .saveAsTable(f"{catalog_name}.silver.slv_date")

### Products Dim Processing

Cleans the products dimension (50,000 rows) by filling null `color` values with 'NA', stripping the `g` suffix from `weight_grams` and casting to float, replacing comma decimal separators in `length_cms`, uppercasing `category_code` and `brand_code`, casting `width_cm` and `height_cms` to float, correcting material spelling mistakes, and applying absolute value to negative `rating_count` values. No duplicates were found.

In [0]:
# Read the products dimension table from the bronze layer
df_prod = spark.read.table(f"{catalog_name}.bronze.brz_products")

df_prod.show(5)

In [0]:
# Check for null values in each column of the products table

df_prod.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df_prod.columns
]).show()

In [0]:
# Check for duplicate product_id values

duplicates = df_prod.groupBy("product_id").count().filter("count > 1")

print(f"Duplicate counts: {duplicates.count()}")
# duplicates.display()

In [0]:
# Analyze the distribution of numeric values using the describe method

df_prod.describe().display()

In [0]:
# Display the row count and column count of the products table

print(f"Row count: {df_prod.count()}")
print(f"Column count: {len(df_prod.columns)}")

In [0]:
# Verify that all product_id values are unique by comparing distinct count with total count
distinct_ids = df_prod.select("product_id").distinct().count()
total_rows = df_prod.count()
print(f"Distinct product_id count: {distinct_ids}")
print(f"Total row count: {total_rows}")
print(f"All product_ids are unique: {distinct_ids == total_rows}")

**Observations on the Products table:**

From the schema and description we can identify the following transformations needed:

1. **`product_id` column** — There are no duplicates, so no deduplication is needed.
2. **`color` column** — There are some null values; since this is not a required field, we will replace nulls with `'NA'`.
3. **`weight_grams` column** — Every value has a `g` suffix; we will strip the `g` from the end of each value and cast the column to `float`.
4. **`length_cms` column** — Values use a comma as the decimal separator; we will replace the comma with a dot and cast the column to `float`.
5. **`category_code` and `brand_code` columns** — Convert from lowercase to uppercase.
6. **`width` and `height` columns** — Change the datatype from string to `float`.
7. **`material` column** — There are some spelling mistakes that need to be corrected.
8. **`rating_count` column** — Some values are negative, but all ratings should be positive; we will apply the absolute value to fix this.

In [0]:
# Replace null values in the color column with 'NA'

df_prod = df_prod.withColumn(
    "color",
    F.when(F.col("color").isNull(), "NA").otherwise(F.col("color"))
)

In [0]:
# Strip the 'g' suffix from weight_grams and cast to float

df_prod = df_prod.withColumn(
    "weight_grams", F.regexp_replace(F.col("weight_grams"), "g", "").cast(FloatType())
)

df_prod.select("weight_grams").show(5)

In [0]:
# Replace comma decimal separator with dot in length_cms and cast to float

df_prod = df_prod.withColumn(
    "length_cms", F.regexp_replace(F.col("length_cms"), ',', '.').cast(FloatType())
)

df_prod.select("length_cms").show(5)

In [0]:
# Convert category_code and brand_code from lowercase to uppercase

df_prod = df_prod.withColumn(
    "category_code", F.upper(F.col("category_code"))
).withColumn("brand_code", F.upper(F.col("brand_code")))

df_prod.select(["category_code", "brand_code"]).show(5)

In [0]:
# Cast width and height columns to float type

df_prod = df_prod.withColumn("width_cms", F.col("width_cm").cast(FloatType())) \
                .withColumn('height_cms', F.col('height_cms').cast(FloatType()))

#dropping width_cm column
df_prod = df_prod.drop('width_cm')

df_prod.select(['width_cms', 'height_cms']).show(5)

In [0]:
# Display distinct material values to identify spelling mistakes

df_prod.select("material").distinct().display()

In [0]:
# Correct spelling mistakes in the material column

df_prod = df_prod.withColumn(
    "material",
    F.when(F.col("material") == "Coton", "Cotton")
        .when(F.col("material") == "Alumium", "Aluminum")
        .when(F.col("material") == "Ruber", "Rubber")
        .otherwise(F.col("material"))
)

df_prod.select("material").distinct().display()

In [0]:
# Apply absolute value to rating_count and cast to integer (negative ratings are invalid)

df_prod = df_prod.withColumn(
    "rating_count",
    F.when(F.col("rating_count").isNotNull(), F.abs(F.col("rating_count")).cast(IntegerType())).otherwise(
        F.lit(0).cast(IntegerType())
    ),
)

df_prod.select("rating_count").describe().display()

In [0]:
# Display the final schema of the products table
df_prod.printSchema()

**Save the cleaned Products dimension table to the Silver layer**

All columns in the `brz_products` table have now been cleaned and transformed:

- **`color`** — Replaced null values with `'NA'`.
- **`weight_grams`** — Stripped the `g` suffix and cast to `float`.
- **`length_cms`** — Replaced comma decimal separator with dot and cast to `float`.
- **`category_code` and `brand_code`** — Converted to uppercase for consistency.
- **`width_cms` and `height_cms`** — Cast from string to `float`.
- **`material`** — Corrected spelling mistakes (e.g., `Coton` → `Cotton`, `Alumium` → `Aluminum`, `Ruber` → `Rubber`).
- **`rating_count`** — Applied absolute value to remove negative ratings and cast to `int`.

With the transformations complete, we can write the cleaned DataFrame to the Silver layer as a managed Delta table (`slv_products`).

In [0]:
# Write the cleaned products table to the silver layer

df_prod.write.mode("overwrite") \
            .format("delta") \
            .option("mergeSchema", True) \
            .saveAsTable(f"{catalog_name}.silver.slv_products")

## Summary

All five dimension tables have been cleaned and written to the Silver layer:

| Silver Table | Bronze Rows | Silver Rows | Transformations Applied |
| --- | --- | --- | --- |
| `ecommerce.silver.slv_brands` | 52 | 52 | Removed `@` from brand codes, trimmed whitespace, standardized category codes |
| `ecommerce.silver.slv_category` | 10 | 8 | Uppercased category codes, removed 2 duplicate rows |
| `ecommerce.silver.slv_customers` | 300,000 | 299,700 | Dropped 300 null customer IDs, filled 30K null phones, formatted phone numbers |
| `ecommerce.silver.slv_date` | 95 | 92 | Cast date/year types, title-cased day names, created readable quarter/week labels, removed 3 duplicates |
| `ecommerce.silver.slv_products` | 50,000 | 50,000 | Filled null colors, cast numeric columns, uppercased codes, fixed material spelling, fixed negative ratings |

### Key Data Quality Issues Found and Resolved

* **Brands** — Non-alphanumeric characters (`@`) in brand codes; trailing whitespace in brand names; inconsistent category code formats
* **Category** — Lowercase category codes; duplicate rows for APP and GRCY
* **Customers** — 300 null customer IDs; 30,000 null phone numbers; phone numbers with `.0` suffix from float storage
* **Date** — All columns stored as strings; negative week-of-year values; inconsistent day name casing; 3 duplicate dates
* **Products** — 4,126 null color values; `g` suffix in weight values; comma decimal separators; lowercase codes; misspelled materials (Coton, Alumium, Ruber); negative rating counts